In [0]:
import pyspark as sp 
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
print(spark)
print(spark.version)

In [0]:
import pandas as pd
import numpy as np
pd_temp=pd.DataFrame(np.random.random(10))
spark_temp=spark.createDataFrame(pd_temp)
print(spark.catalog.listTables())
spark_temp.createOrReplaceTempView('temp')
print(spark.catalog.listTables())

In [0]:
file_path=r'/Volumes/project_catalog/bronze/volume/airports.csv'
airports=spark.read.csv(file_path,header=True)
airports.show()

In [0]:
type(airports)

In [0]:
spark.catalog.listDatabases()

In [0]:
spark.catalog.listTables()

In [0]:
flights=spark.read.csv('/Volumes/project_catalog/bronze/volume/flights_small.csv', header=True)
flights.show()

In [0]:
flights.name=flights.createOrReplaceTempView('flights')
spark.catalog.listTables()


In [0]:
flights_df=spark.table('flights')
print(flights_df.show())

In [0]:
from pyspark.sql.functions import expr
flights=flights.withColumn("duration_hrs", expr("try_cast(air_time as double)/ 60"))
flights.show()

In [0]:
flights.describe().show()

In [0]:
long_flights1=flights.filter("distance>1000")
long_flights1.show()

In [0]:
long_flights2=flights.filter(flights.distance>1000)
long_flights2.show()

In [0]:
selected_1=flights.select('tailnum', 'origin', 'dest')
selected_1.show()

In [0]:
temp=flights.select(flights.origin, flights.dest, flights.carrier)



In [0]:
FilterA=flights.origin=='SEA'
FilterB=flights.dest=='PDX'
selected_2=temp.filter(FilterA).filter(FilterB)
selected_2.show()

In [0]:
avg_speed=(flights.distance/(flights.air_time/60)).alias("avg_speed")
speed_1=flights.select('origing', 'dest', 'tailnum', avg_speed)


In [0]:
speed_2=flights.selectExpr('origin', 'dest', 'tailnum', 'distance/(try_cast(air_time as double)/60) as avg_speed')
speed_2.show()

In [0]:
flights.describe()

In [0]:
flights=flights.withColumn('distance', expr('try_cast(distance as float)'))

flights=flights.withColumn('air_time', expr('try_cast(air_time as float)'))

flights.describe('air_time', 'distance').show()


In [0]:
flights.filter(flights.origin=='PDX').groupby().min('distance').show()

In [0]:
flights.filter(flights.origin=='SEA').groupBy().max('air_time').show()

In [0]:
flights.filter(flights.carrier=='DL').filter(flights.origin=='SEA').groupBy().avg('air_time').show()


In [0]:
flights.withColumn('duration_hrs', flights.air_time/60).groupBy().sum('duration_hrs').show()

In [0]:
by_plane=flights.groupBy('tailnum')
by_plane.count().show()

In [0]:
by_origin=flights.groupBy('origin')
by_origin.avg('air_time').show()

In [0]:
import pyspark.sql.functions as F
flights=flights.withColumn('dep_delay', F.expr('try_cast(dep_delay as integer)'))
by_month_dest=flights.groupBy('month', 'dest')

                           

In [0]:
by_month_dest.avg('dep_delay').show()

In [0]:
airports.show()

In [0]:
airports=airports.withColumnRenamed('faa', 'dest')

In [0]:
flights_with_airports=flights.join(airports, on='dest', how='leftouter')
flights_with_airports.show()

In [0]:
planes=spark.read.csv('/Volumes/project_catalog/bronze/volume/planes.csv', header=True)
planes.show()

In [0]:
planes=planes.withColumnRenamed('year', 'plane_year')

In [0]:
model_data=flights.join(planes, on='tailnum', how='leftouter')
model_data.show()

In [0]:
model_data.describe()

In [0]:
model_data=model_data.withColumn('arr_delay', F.expr('try_cast(arr_delay as integer)'))
model_data=model_data.withColumn('air_time', F.expr('try_cast(air_time as integer)'))
model_data=model_data.withColumn('month', F.expr('try_cast(month as integer)'))
model_data=model_data.withColumn('plane_year', F.expr('try_cast (plane_year as integer)'))

In [0]:
model_data.describe('arr_delay', 'air_time', 'month', 'plane_year').show()

In [0]:
model_data=model_data.withColumn('plane_age', model_data.year-model_data.plane_year)


In [0]:
model_data=model_data.withColumn('is_late', model_data.arr_delay>0)
model_data=model_data.withColumn('label', F.expr('try_cast(is_late as integer)'))
model_data.filter("arr_delay is not NULL and dep_delay is not NULL and air_time is not NULL and plane_year is not NULL")

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
carr_indexer=StringIndexer(inputCol='carrier', outputCol='carrier_index')
carr_encoder=OneHotEncoder(inputCol='carrier_index', outputCol='carr_fact')

In [0]:
dest_indexer=StringIndexer(inputCol='dest', outputCol='dest_index')
dest_encoder=OneHotEncoder(inputCol='dest_index', outputCol='dest_fact')


In [0]:
from pyspark.ml.feature import VectorAssembler
vec_assembler=VectorAssembler(inputCols=['month', 'air_time', 'carr_fact', 'plane_age'], 
outputCol='features', handleInvalid="skip")

In [0]:
from pyspark.ml import Pipeline

flights_pipe=Pipeline(stages=[dest_indexer, dest_encoder, carr_indexer, carr_encoder, vec_assembler])

piped_data=flights_pipe.fit(model_data).transform(model_data)

In [0]:
piped_data.show()

In [0]:
training, test=piped_data.randomSplit([.6, .4])

In [0]:
from pyspark.ml.classification import LogisticRegression
lr=LogisticRegression()

In [0]:
import pyspark.ml.evaluation as evals
evaluator=evals.BinaryClassificationEvaluator(metricName='areaUnderROC')

In [0]:
import pyspark.ml.tuning as tune

grid=tune.ParamGridBuilder()

grid=grid.addGrid(lr.regParam, np.arange(0, .1, .01))
grid=grid.addGrid(lr.elasticNetParam, [0, 1])

grid=grid.build()

In [0]:
cv=tune.CrossValidator(estimator=lr, 
                       estimatorParamMaps=grid,
                       evaluator=evaluator)

In [0]:
import os
os.environ['SPARKML_TEMP_DFS_PATH']='/Volumes/project_catalog/bronze/volume'
models=cv.fit(training)

In [0]:
best_lr=models.bestModel

In [0]:
test_results=best_lr.transform(test)
print(evaluator.evaluate(test_results))